# Run the backend on Colab GPU

Runs this repo's FastAPI backend (`backend/`) on a Colab GPU runtime and exposes it to the
internet with an ngrok tunnel, so your **local** frontend (`npm run dev` on your own machine,
unchanged) can call it instead of a slow local CPU backend.

**Before running:**
1. `Runtime > Change runtime type > GPU`, then `Runtime > Restart session` if you changed it.
2. One-time only: upload `backend/models/best_augmented.pt` and everything in `backend/data/`
   (`class_keywords.json`, `embeddings.json`, ...) from your machine into a Google Drive folder
   at `MyDrive/lipton-sku-classifier-assets/models/` and `MyDrive/lipton-sku-classifier-assets/data/`
   respectively. These files are gitignored (see `backend/.gitignore`) so they don't come from
   `git clone` below — Drive is just the hand-off point between your machine and the Colab VM.
3. One-time only: accept the license for the gated `facebook/dinov3-vitb16-pretrain-lvd1689m`
   model on Hugging Face and grab an access token — see the "Hugging Face login" cell below for
   the exact steps.
4. One-time only: sign up for a free ngrok account and grab an authtoken — see the "ngrok auth"
   cell below for the exact steps.
5. Run every cell top to bottom. The last cell prints the URL to paste into your local
   `frontend/.env` as `VITE_API_BASE`.

The backend and tunnel keep running as background processes as long as this notebook's runtime
stays alive — closing the tab eventually disconnects the runtime (Colab free tier), which kills
both, so re-run the notebook to get a new session and a new tunnel URL.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError(
        "No GPU detected. Go to Runtime > Change runtime type > GPU, restart, and re-run."
    )

In [ ]:
REPO_URL = "https://github.com/ansshahzadd/lipton-sku-classifier.git"
REPO_DIR = "/content/lipton-sku-classifier"
BACKEND_DIR = f"{REPO_DIR}/backend"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

In [ ]:
# Install everything from requirements.txt (this pulls in the CPU paddlepaddle build), then
# try to swap in the matching GPU build so OCR can run on the GPU too. paddlepaddle-gpu is NOT
# on PyPI -- it only exists on Baidu's own package index, one wheel per CUDA version -- so a
# plain `pip install paddlepaddle-gpu` always 404s there. We detect Colab's CUDA version via
# torch and hit the right index; if nothing matches, we keep the CPU paddle and just run OCR on
# CPU (YOLO + DINOv3 still get the full GPU speedup either way).
%pip install -q -r {BACKEND_DIR}/requirements.txt

import subprocess
import torch

CUDA_TAG_MAP = {"11.8": "cu118", "12.6": "cu126", "12.9": "cu129"}
cuda_version = torch.version.cuda
cuda_tag = CUDA_TAG_MAP.get(cuda_version)
print("torch reports CUDA", cuda_version, "-> paddle wheel tag:", cuda_tag)

paddle_gpu_ok = False
if cuda_tag:
    result = subprocess.run(
        ["pip", "install", "-q", "paddlepaddle-gpu",
         "-i", f"https://www.paddlepaddle.org.cn/packages/stable/{cuda_tag}/"],
        capture_output=True, text=True,
    )
    paddle_gpu_ok = result.returncode == 0
    if not paddle_gpu_ok:
        print(result.stdout[-2000:])
        print(result.stderr[-2000:])

print("paddlepaddle-gpu installed:", paddle_gpu_ok, "-> OCR will run on", "GPU" if paddle_gpu_ok else "CPU")

In [ ]:
# DINOv3 (facebook/dinov3-vitb16-pretrain-lvd1689m) is a GATED model on Hugging Face -- pipeline.py
# downloads it on first run, and that download 401s until you've (1) accepted its license and
# (2) authenticated here with a token. One-time setup, from your own machine/browser:
#   1. Log into https://huggingface.co (free account is fine).
#   2. Open https://huggingface.co/facebook/dinov3-vitb16-pretrain-lvd1689m and click
#      "Agree and access repository" (Meta's DINOv3 grants are usually instant).
#   3. Create a read-scope token at https://huggingface.co/settings/tokens.
# Then either: add it as a Colab secret named HF_TOKEN (key icon in the left sidebar, toggle
# "Notebook access" on) so this cell picks it up with no prompt, or just paste it when asked
# below (input is hidden either way, nothing gets printed or saved into this notebook).
import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if not hf_token:
    import getpass
    hf_token = getpass.getpass("Hugging Face token (read scope): ")

login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token
print("Logged in to Hugging Face.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

ASSETS_DIR = "/content/drive/MyDrive/lipton-sku-classifier-assets"

for sub in ("models", "data"):
    src = os.path.join(ASSETS_DIR, sub)
    dst = os.path.join(BACKEND_DIR, sub)
    if not os.path.isdir(src):
        raise FileNotFoundError(
            f"Expected {src} on your Drive. Upload your {sub}/ files there first (see the "
            "instructions in the first cell), then re-run this cell."
        )
    os.makedirs(dst, exist_ok=True)
    for fname in os.listdir(src):
        shutil.copy2(os.path.join(src, fname), os.path.join(dst, fname))
    print(f"{sub}/ ->", os.listdir(dst))

In [ ]:
%pip install -q pyngrok

In [ ]:
from pyngrok import ngrok

# Kill any tunnel left over from a previous run of this cell.
ngrok.kill()

tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url.replace("http://", "https://")
print("Tunnel URL:", public_url)

In [ ]:
import re, subprocess, threading, time

# Kill any tunnel left over from a previous run of this cell -- otherwise re-running it just
# piles up duplicate cloudflared processes all pointing at the same local port.
if "tunnel_proc" in globals():
    try:
        tunnel_proc.terminate()
        tunnel_proc.wait(timeout=5)
    except Exception:
        pass
!pkill -f "cloudflared tunnel" 2>/dev/null || true
time.sleep(1)

tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
_url_pattern = re.compile(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com")

def _watch_tunnel_output():
    global public_url
    for line in tunnel_proc.stdout:
        match = _url_pattern.search(line)
        if match and public_url is None:
            public_url = match.group(0)
        print(line, end="")

threading.Thread(target=_watch_tunnel_output, daemon=True).start()

for _ in range(60):
    if public_url:
        break
    time.sleep(1)

if not public_url:
    raise RuntimeError("No tunnel URL after 60s — check the cloudflared output above for errors.")

print("\nTunnel URL:", public_url)

In [ ]:
import os, subprocess, threading, time

# Kill any backend left over from a previous run of this cell -- otherwise the new uvicorn
# fails to bind port 8000 ("address already in use") and silently exits, while the old,
# possibly-still-busy process keeps eating every request.
if "backend_proc" in globals():
    try:
        backend_proc.terminate()
        backend_proc.wait(timeout=5)
    except Exception:
        pass
!pkill -f "uvicorn main:app" 2>/dev/null || true
time.sleep(1)

env = os.environ.copy()
env["PUBLIC_BASE_URL"] = public_url
env["FRONTEND_ORIGINS"] = "http://localhost:5173,http://127.0.0.1:5173"
env["OCR_DEVICE"] = "gpu" if paddle_gpu_ok else "cpu"

backend_proc = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=BACKEND_DIR,
    env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

def _watch_backend_output():
    for line in backend_proc.stdout:
        print(line, end="")

threading.Thread(target=_watch_backend_output, daemon=True).start()
print("uvicorn starting (pid", backend_proc.pid, ") — first request will be slow while models load.")

## Notes

- OCR device is picked automatically: the install cell tries to match Colab's CUDA version to a
  `paddlepaddle-gpu` wheel from Baidu's package index and sets `paddle_gpu_ok` accordingly; the
  uvicorn cell reads that to set `OCR_DEVICE`. If it falls back to CPU, YOLO + DINOv3 still get
  the full GPU speedup — only OCR runs on CPU.
- We use ngrok instead of Cloudflare's quick tunnel because `*.trycloudflare.com` force-closes
  any single request after ~100s, which a slow multi-crop shelf-image upload can easily exceed;
  ngrok doesn't impose that cap. The frontend's `api.js` sends the
  `ngrok-skip-browser-warning` header so ngrok's free-tier interstitial page doesn't intercept
  the API calls — don't remove that header if you're using ngrok.
- ngrok's tunnel URL is random and changes every time you re-run the tunnel cell — update
  `VITE_API_BASE` locally whenever you restart this notebook.
- Re-running the tunnel or uvicorn cell kills whatever it previously started first, so it's
  safe to re-run either one on its own without restarting the whole runtime.
- A single `/api/images` upload can legitimately take a while (many crops per shelf photo, each
  doing an embedding + up to 2 OCR passes) — `main.py` runs it in a background thread
  (`asyncio.to_thread`) so it no longer freezes the whole server for other requests while it's
  in flight, but the upload itself is still just as slow.
- To stop everything: `backend_proc.terminate(); ngrok.kill()`, or just stop/disconnect the
  Colab runtime.

## Notes

- OCR device is picked automatically: the install cell tries to match Colab's CUDA version to a
  `paddlepaddle-gpu` wheel from Baidu's package index and sets `paddle_gpu_ok` accordingly; the
  uvicorn cell reads that to set `OCR_DEVICE`. If it falls back to CPU, YOLO + DINOv3 still get
  the full GPU speedup — only OCR runs on CPU.
- Cloudflare's quick tunnel URL is random and changes every time you re-run the tunnel cell —
  update `VITE_API_BASE` locally whenever you restart this notebook.
- Re-running the tunnel or uvicorn cell kills whatever it previously started first, so it's
  safe to re-run either one on its own without restarting the whole runtime.
- A single `/api/images` upload can legitimately take a while (many crops per shelf photo, each
  doing an embedding + up to 2 OCR passes) — `main.py` now runs it in a background thread
  (`asyncio.to_thread`) so it no longer freezes the whole server for other requests while it's
  in flight, but the upload itself is still just as slow.
- To stop everything: `backend_proc.terminate(); tunnel_proc.terminate()`, or just stop/disconnect
  the Colab runtime.